# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds
**Nguyễn Thế Khang — 2A202602964** · Google Colab, GPU T4

Notebook này chạy **toàn bộ** bài lab bằng code trong `code/` (một hàm `train.run(Config)` cho mọi thí nghiệm):
Bước 0 (EDA + kiểm tra) → Bước 1 (7 backbone + DINOv2 probe) → Bước 2 (14 ablation + 2 kết hợp, T00 × 3 seed)
→ Bước 3 (suy luận + độ trễ) → Bước 4 (chung kết 3 seed, **test một lần mỗi seed**, `eval.py`) → Bước 5 (xlsx, báo cáo).

**Chạy lại an toàn:** mọi kết quả ghi ra Google Drive (`WORK`). Nếu Colab ngắt, chạy lại từ đầu: lần chạy đã xong được
bỏ qua, lần đang dở tiếp tục từ epoch cuối. Mọi lựa chọn là hàm định trước của số liệu **val** (`code/experiments.py`).

## 0. Môi trường, mã nguồn, dữ liệu

In [ ]:
# Runtime > Change runtime type > T4 GPU
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/deepweeds_lab_work'          # kết quả bền (runs, predictions, logs)
REPO_URL = 'https://github.com/khangnguyenthe18/K4-DAY02-NguyenTheKhang-2A202602964.git'
SUB = 'submissions/2A202602964_NguyenTheKhang'

import os, subprocess
if not os.path.exists('/content/lab'):
    subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/lab'], check=True)
else:
    subprocess.run(['git', '-C', '/content/lab', 'pull', '-q'], check=False)
CODE = f'/content/lab/{SUB}/code'
os.makedirs(WORK, exist_ok=True)
!pip -q install timm fvcore openpyxl

In [ ]:
import sys, platform
sys.path.insert(0, CODE); sys.path.insert(0, '/content/lab')     # code/ và eval.py gốc
import numpy as np, pandas as pd, torch, timm, torchvision
print('python', platform.python_version(), '| torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')
assert torch.cuda.is_available(), 'Bật GPU T4 trong Runtime > Change runtime type'

In [ ]:
# Tải dữ liệu vào đĩa cục bộ của Colab (nhanh hơn đọc từ Drive). Kiểm tra MD5.
import hashlib
DATA = '/content/data'
os.makedirs(f'{DATA}/labels', exist_ok=True)
if not os.path.exists(f'{DATA}/images.zip'):
    !wget -q -O {DATA}/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"
h = hashlib.md5()
with open(f'{DATA}/images.zip', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == 'b7b30f96d466fba86016aa5a26606e0f', f'MD5 sai: {h.hexdigest()}'
if not os.path.exists(f'{DATA}/images'):
    !unzip -q -n {DATA}/images.zip -d {DATA}/images
BASE = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ['labels', 'train_subset0', 'val_subset0', 'test_subset0']:
    if not os.path.exists(f'{DATA}/labels/{name}.csv'):
        !wget -q -O {DATA}/labels/{name}.csv {BASE}/{name}.csv
# tìm đúng thư mục chứa .jpg sau khi giải nén
IMAGES_DIR = next(r for r, _, fs in os.walk(f'{DATA}/images') if sum(f.endswith('.jpg') for f in fs) > 17000)
LABELS_DIR = f'{DATA}/labels'
print(IMAGES_DIR, len(os.listdir(IMAGES_DIR)), 'ảnh')

In [ ]:
from experiments import Paths
import experiments as X, pipeline as P
paths = Paths(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, cache_dir=f'{DATA}/cache', work_dir=WORK, num_workers=2)
# Kiểm tra code trước khi tốn GPU: unit test tự viết (focal γ=0 ≡ CE, CutMix, gộp BN, ...)
!cd {CODE} && python -m unittest discover -s tests 2>&1 | tail -3

## Bước 0 — EDA và kiểm tra pipeline
Kiểm tra chia dữ liệu (README 2.1: số ảnh, giao rỗng, hợp = 17.509, file tồn tại), phân bố lớp so với Table 1,
ảnh mẫu, loss ban đầu ≈ ln 9 cho mọi backbone, overfit một batch, ảnh sau augmentation và CutMix/Mixup.

In [ ]:
P.step0(paths)
import json
from IPython.display import Image, display
s0 = json.load(open(f'{WORK}/step0.json'))
print(json.dumps(s0['eda']['split_check'], indent=1)); print(json.dumps(s0['sanity']['initial_loss'], indent=1))
print(s0['sanity']['overfit'])
display(pd.DataFrame(s0['eda']['counts']))
for f in ['eda_class_distribution', 'eda_samples', 'sanity_overfit_one_batch', 'sanity_augmentations', 'sanity_mix']:
    display(Image(f'{WORK}/figures/{f}.png', width=900))

## Bước 1 — 7 backbone, cùng công thức nền T00, seed 0 (+ DINOv2 linear probe, điểm thưởng)
ResNet-50, ResNeXt-50, ConvNeXt-T, DeiT-S, Swin-T, EfficientNet-B0, MobileNetV3-L. Chọn backbone theo quy tắc định trước:
macro-F1 val cao nhất, các backbone cách ≤ 0,005 coi là ngang nhau → chọn cái có độ trễ thấp nhất.

In [ ]:
P.step1(paths, with_dinov2=True)
from make_results import all_runs, sheet_backbones
display(sheet_backbones(paths, all_runs(paths), X.load_decisions(paths)))
print(X.load_decisions(paths)['backbone']['reason'])

## Bước 2 — Công thức huấn luyện
T00 × 3 seed (đo nhiễu) + 14 ablation **một yếu tố** (trục A khởi tạo, B augmentation, C loss, D sampler, E LR, F EMA,
G độ phân giải/epoch) + 2 kết hợp định trước (T15: yếu tố có Δ > std; T16: yếu tố có Δ > 0).

In [ ]:
P.step2(paths)
from make_results import sheet_training
display(sheet_training(paths, all_runs(paths), X.load_decisions(paths)))
print(X.load_decisions(paths)['recipe']['reason'])

## Bước 3 — Suy luận (trên val, không huấn luyện lại) và độ trễ
TTA lật, 10-crop, đa tỉ lệ, gộp xác suất vs logit, dò độ phân giải, ensemble, soup, EMA, temperature scaling + ECE,
gộp BN, FP16/AMP/channels_last. Độ trễ: warmup 10, `cuda.synchronize`, 100 lần đo, p50/p95/p99, batch 1 và 32.

In [ ]:
P.step3(paths)
from make_results import sheet_inference
display(sheet_inference(json.load(open(f'{WORK}/step3.json'))))
print(X.load_decisions(paths)['inference']['reason'])

## Bước 4 — Chung kết: 3 seed mới, **test một lần mỗi seed**
Huấn luyện lại công thức đã chốt với seed 10, 11, 12; mốc là T00 seed 0–2 + I00. Mỗi lần đọc test ghi vào
`runs/test_access.log`; lần thứ hai bị từ chối. Sau đó chạy `eval.py score` và `eval.py grade` gốc (ô dưới in lại).

In [ ]:
P.step4(paths)

In [ ]:
# Chỉ số theo đúng định nghĩa của lớp (eval.py gốc, không sửa)
PRED = f'{WORK}/predictions'
!python /content/lab/eval.py score --pred "{PRED}/F01_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out {WORK}/eval_out
!python /content/lab/eval.py score --pred "{PRED}/T00_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out {WORK}/eval_out

In [ ]:
fin = json.load(open(f'{WORK}/final.json'))
p95 = fin['realtime_latency']['p95_ms']
!python /content/lab/eval.py grade --final "{PRED}/F01_seed*_test.csv" --baseline "{PRED}/T00_seed*_test.csv" --uncal "{PRED}/F01uncal_seed*_test.csv" --final-val "{PRED}/F01_seed*_val.csv" --val-csv {LABELS_DIR}/val_subset0.csv --latency-p95-ms {p95} --latency-method proper --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out {WORK}/eval_out

## Bước 5 — results.xlsx, curves/, figures/, report.md, predictions/ → tải về và đặt vào thư mục bài nộp

In [ ]:
OUT = f'{WORK}/submission_out'
P.step5(paths, __import__('pathlib').Path(OUT))
!cd {OUT} && zip -qr {WORK}/submission_outputs.zip results.xlsx report.md curves figures predictions eval_out logs
print('Tải file này về:', f'{WORK}/submission_outputs.zip')
from google.colab import files
files.download(f'{WORK}/submission_outputs.zip')